In [ ]:
import re
import numpy as np, pandas as pd
from pathlib import Path

D = Path(r"C:\Users\arz_2\Basira_Motassim_Group_A\validation_data\OMNICS_CPTAC_GC")      # edit: folder with the downloaded files
CAD = ["CDH1","CDH2","CDH3","CDH11","CDH13","CDH17","CDHR2","CDHR5",
       "CLSTN1","DSC2","DSG2","FAT1","PCDH1","PCDH17","PCDH18","PCDHGA10"]
base = lambda s: re.sub(r"\.[TN]$", "", str(s))

def read_feat(path):
    df = pd.read_csv(path, sep="\t", index_col=0, low_memory=False)
    if any(re.match(r"C3[LN]", str(c)) for c in df.columns[:5]):   # samples in columns -> flip
        df = df.T
    return df

mut = pd.read_csv(D / "GC_WES_gene-level-somatic-mutation-binary_cbt.txt", sep="\t", index_col=0)
mut.columns = [base(c) for c in mut.columns]

RNA = D / "GC_RNAseq_gene_rsem_rm_circRNA_UQ_log2(x+1)_Tumor.cct"
rna_ids = {base(c) for c in pd.read_csv(RNA, sep="\t", nrows=0, index_col=0).columns}

clin = read_feat(D / "GC_Jan2025_clinical_data.tsi")
clin.index = [base(i) for i in clin.index]

print("example IDs -> mut:", list(mut.columns[:2]), "| rna:", sorted(rna_ids)[:2], "| clin:", list(clin.index[:2]))
print("duplicate clinical IDs:", int(clin.index.duplicated().sum()))
print("clinical shape:", clin.shape)
print("clinical columns:", list(clin.columns), "\n")

common = sorted(set(mut.columns) & rna_ids & set(clin.index))
print("mutation:", mut.shape[1], "| RNA:", len(rna_ids), "| clinical:", clin.shape[0],
      "| in all three:", len(common))

tp  = mut.loc["TP53", common].astype(bool)
cad = mut.loc[CAD, common].sum() > 0
grp = pd.Series(np.select([tp & cad, tp, cad], ["Co-mutated", "TP53 only", "Cadherin only"], "Wild-type"),
                index=common, name="Mutation_Group")
print(grp.value_counts())
grp.to_csv("CPTAC_groups.csv")